In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, log_loss, brier_score_loss

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Torch version: {torch.__version__}")


Using device: mps
Torch version: 2.10.0


In [2]:
model_name = "cross-encoder/nli-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Model num_labels: {model.config.num_labels}")
print(f"Model id2label: {getattr(model.config, 'id2label', None)}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: cross-encoder/nli-distilroberta-base
Model num_labels: 3
Model id2label: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])


Dataset split: glue/mrpc validation
Number of examples: 408
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
batch_size = 32
max_length = 128

labels = np.array(dataset["label"])
predictions = []
confidences = []
positive_probs = []
full_probs = []
raw_nli_probs = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        nli_probs = torch.softmax(logits, dim=-1)

        if logits.shape[-1] == 3:
            contradiction = nli_probs[:, 0]
            entailment = nli_probs[:, 2]
            pair_logits = torch.stack([contradiction, entailment], dim=-1)
            pair_probs = pair_logits / pair_logits.sum(dim=-1, keepdim=True)
        elif logits.shape[-1] == 2:
            pair_probs = nli_probs
        else:
            raise ValueError(f"Unexpected number of labels: {logits.shape[-1]}")

        preds = torch.argmax(pair_probs, dim=-1)
        confs = pair_probs.max(dim=-1).values

    predictions.extend(preds.cpu().tolist())
    confidences.extend(confs.cpu().tolist())
    positive_probs.extend(pair_probs[:, 1].cpu().tolist())
    full_probs.extend(pair_probs.cpu().tolist())
    raw_nli_probs.extend(nli_probs.cpu().tolist())

predictions = np.array(predictions)
confidences = np.array(confidences)
positive_probs = np.array(positive_probs)
full_probs = np.array(full_probs)
raw_nli_probs = np.array(raw_nli_probs)

print(f"Completed inference for {len(predictions)} examples.")
print(f"Binary paraphrase probability matrix shape: {full_probs.shape}")
print(f"Raw NLI probability matrix shape: {raw_nli_probs.shape}")


Completed inference for 408 examples.
Binary paraphrase probability matrix shape: (408, 2)
Raw NLI probability matrix shape: (408, 3)


In [5]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)
nll = log_loss(labels, full_probs, labels=[0, 1])
brier = brier_score_loss(labels, positive_probs)

correct = (predictions == labels).astype(int)
n_bins = 10
bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
ece = 0.0
bin_summaries = []

for i in range(n_bins):
    left = bin_edges[i]
    right = bin_edges[i + 1]
    if i == n_bins - 1:
        mask = (confidences >= left) & (confidences <= right)
    else:
        mask = (confidences >= left) & (confidences < right)

    count = int(mask.sum())
    if count > 0:
        bin_acc = float(correct[mask].mean())
        bin_conf = float(confidences[mask].mean())
        gap = abs(bin_acc - bin_conf)
        pos_rate = float(labels[mask].mean())
        pred_pos_rate = float(predictions[mask].mean())
        avg_pos_prob = float(positive_probs[mask].mean())
        ece += (count / len(labels)) * gap
        bin_summaries.append({
            "bin": i,
            "range": f"[{left:.1f}, {right:.1f}]" if i == n_bins - 1 else f"[{left:.1f}, {right:.1f})",
            "count": count,
            "avg_confidence": bin_conf,
            "accuracy": bin_acc,
            "gap": gap,
            "true_positive_rate": pos_rate,
            "pred_positive_rate": pred_pos_rate,
            "avg_paraphrase_prob": avg_pos_prob,
        })
    else:
        bin_summaries.append({
            "bin": i,
            "range": f"[{left:.1f}, {right:.1f}]" if i == n_bins - 1 else f"[{left:.1f}, {right:.1f})",
            "count": 0,
            "avg_confidence": None,
            "accuracy": None,
            "gap": None,
            "true_positive_rate": None,
            "pred_positive_rate": None,
            "avg_paraphrase_prob": None,
        })

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"NLL      : {nll:.4f}")
print(f"Brier    : {brier:.4f}")
print(f"ECE      : {ece:.4f}")
print("Confusion matrix:")
print(cm)

print("\nCalibration bin summary:")
for summary in bin_summaries:
    if summary["count"] == 0:
        print(f"bin={summary['bin']:2d} range={summary['range']:>11} count=0")
    else:
        print(
            f"bin={summary['bin']:2d} range={summary['range']:>11} count={summary['count']:3d} "
            f"avg_conf={summary['avg_confidence']:.4f} acc={summary['accuracy']:.4f} "
            f"gap={summary['gap']:.4f} avg_p1={summary['avg_paraphrase_prob']:.4f} "
            f"true_p1={summary['true_positive_rate']:.4f} pred_p1={summary['pred_positive_rate']:.4f}"
        )


Evaluation metrics:
Accuracy : 0.6642
Precision: 0.7006
Recall   : 0.8889
F1       : 0.7836
NLL      : 1.1724
Brier    : 0.2924
ECE      : 0.2303
Confusion matrix:
[[ 23 106]
 [ 31 248]]

Calibration bin summary:
bin= 0 range= [0.0, 0.1) count=0
bin= 1 range= [0.1, 0.2) count=0
bin= 2 range= [0.2, 0.3) count=0
bin= 3 range= [0.3, 0.4) count=0
bin= 4 range= [0.4, 0.5) count=0
bin= 5 range= [0.5, 0.6) count= 24 avg_conf=0.5482 acc=0.7083 gap=0.1601 avg_p1=0.5367 true_p1=0.7917 pred_p1=0.7500
bin= 6 range= [0.6, 0.7) count= 37 avg_conf=0.6484 acc=0.6486 gap=0.0003 avg_p1=0.5558 true_p1=0.8649 pred_p1=0.6757
bin= 7 range= [0.7, 0.8) count= 47 avg_conf=0.7638 acc=0.7660 gap=0.0021 avg_p1=0.6063 true_p1=0.7234 pred_p1=0.7021
bin= 8 range= [0.8, 0.9) count= 63 avg_conf=0.8506 acc=0.6667 gap=0.1839 avg_p1=0.7710 true_p1=0.6825 pred_p1=0.8889
bin= 9 range= [0.9, 1.0] count=237 avg_conf=0.9723 acc=0.6414 gap=0.3310 avg_p1=0.9154 true_p1=0.6371 pred_p1=0.9367


/Users/jinjinzhao/Documents/work_projects/tablevault_experiments/tablevault_experiments/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [6]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
correct_indices = np.where(predictions == labels)[0]
incorrect_indices = np.where(predictions != labels)[0]

lowest_conf_correct_idx = correct_indices[np.argmin(confidences[correct_indices])] if len(correct_indices) > 0 else None
highest_conf_incorrect_idx = incorrect_indices[np.argmax(confidences[incorrect_indices])] if len(incorrect_indices) > 0 else None
best_prob_correct_idx = correct_indices[np.argmax(confidences[correct_indices])] if len(correct_indices) > 0 else None
worst_prob_incorrect_idx = incorrect_indices[np.argmin(confidences[incorrect_indices])] if len(incorrect_indices) > 0 else None

def print_example(idx, title):
    row = dataset[int(idx)]
    true_label = int(labels[idx])
    pred_label = int(predictions[idx])
    conf = float(confidences[idx])
    pos_prob = float(positive_probs[idx])
    neg_prob = float(full_probs[idx][0])
    print(title)
    print(f"index      : {idx}")
    print(f"sentence1  : {row['sentence1']}")
    print(f"sentence2  : {row['sentence2']}")
    print(f"true label : {true_label} ({label_map[true_label]})")
    print(f"pred label : {pred_label} ({label_map[pred_label]})")
    print(f"confidence : {conf:.4f}")
    print(f"p(class=0) : {neg_prob:.4f}")
    print(f"p(class=1) : {pos_prob:.4f}")
    if raw_nli_probs.shape[1] == 3:
        print(f"p(contradiction): {float(raw_nli_probs[idx][0]):.4f}")
        print(f"p(neutral)      : {float(raw_nli_probs[idx][1]):.4f}")
        print(f"p(entailment)   : {float(raw_nli_probs[idx][2]):.4f}")
    print("-" * 80)

if lowest_conf_correct_idx is not None:
    print_example(lowest_conf_correct_idx, "Lowest-confidence correct example")
else:
    print("No correct predictions found.")

if highest_conf_incorrect_idx is not None:
    print_example(highest_conf_incorrect_idx, "Highest-confidence incorrect example")
else:
    print("No incorrect predictions found.")

if best_prob_correct_idx is not None:
    print_example(best_prob_correct_idx, "Highest-confidence correct example")

if worst_prob_incorrect_idx is not None:
    print_example(worst_prob_incorrect_idx, "Lowest-confidence incorrect example")


Lowest-confidence correct example
index      : 116
sentence1  : Waiting crowds filling the streets on both sides overwhelmed the peacekeepers soon after daylight , sweeping past the barbed wire barricades .
sentence2  : But waiting crowds filling the streets rushed the bridges soon after daylight , overrunning razor-wire barricades .
true label : 1 (paraphrase)
pred label : 1 (paraphrase)
confidence : 0.5035
p(class=0) : 0.4965
p(class=1) : 0.5035
p(contradiction): 0.2132
p(neutral)      : 0.5706
p(entailment)   : 0.2162
--------------------------------------------------------------------------------
Highest-confidence incorrect example
index      : 37
sentence1  : The civilian unemployment rate improved marginally last month -- slipping to 6.1 percent -- even as companies slashed payrolls by 93,000 .
sentence2  : The civilian unemployment rate improved marginally last month _ sliding down to 6.1 percent _ as companies slashed payrolls by 93,000 amid continuing mixed signals about the 

In [7]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"batch_size={batch_size}")
print(f"max_length={max_length}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"nll={nll:.4f}")
print(f"brier={brier:.4f}")
print(f"ece={ece:.4f}")


RESULT SUMMARY
model=cross-encoder/nli-distilroberta-base
dataset_split=glue/mrpc validation
device=mps
num_examples=408
batch_size=32
max_length=128
accuracy=0.6642
precision=0.7006
recall=0.8889
f1=0.7836
nll=1.1724
brier=0.2924
ece=0.2303
